# ELITE vs HighCap — Single Test Year Side-by-Side

Focused comparison of two market-picking methods for one specific test year.

- **ELITE** — analog search with `elite_top_pct` filter: cosine similarity to top-decile-of-top-decile historical winners
- **HighCap** — pure heuristic: rank by ACS `cap_rate_proxy`, take top-N

Both use the same universe filter and same state cap. Shows picks side-by-side + intersection + realized returns (if the test year is observable).

**Edit the config in Section 0, then Kernel → Restart & Run All.**

## 0. Configuration

In [14]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '/Users/premputtegowda/reps/reps-tracker/emerging_re')

import duckdb, numpy as np, pandas as pd

WAREHOUSE = "/Users/premputtegowda/reps/reps-tracker/emerging_re/data/warehouse/emerging_re.duckdb"

# ⚙️ EDIT ME
TEST_YEAR            = 2020          # cohort year to score picks for (2015-2025)
POP_MIN, POP_MAX     = 300_000, 500_000
DIVERSITY_HHI_MAX    = 0.10
TOP_N                = 10
MAX_PER_STATE        = 4
ELITE_TOP_PCT        = 0.25

# Fixed
HORIZON_YEARS        = 3
STRICT_PIT           = True          # honest walk-forward

pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 60)

print(f"Test year:  {TEST_YEAR}")
print(f"Universe:   pop {POP_MIN//1000}K-{POP_MAX//1000}K   HHI ≤ {DIVERSITY_HHI_MAX}")
print(f"Picking:    top-{TOP_N}   max {MAX_PER_STATE}/state   ELITE = top {ELITE_TOP_PCT*100:.0f}% of winners")

Test year:  2020
Universe:   pop 300K-500K   HHI ≤ 0.1
Picking:    top-10   max 4/state   ELITE = top 25% of winners


## 1. Load panel + run ELITE model

In [15]:
from backtest.analog_search import (
    load_tertiary_panel, load_mf_context, build_acs_target,
    build_feature_matrix, zscore_within_cohort, walk_forward,
    TARGET_COL_TOTAL_RETURN,
)

# Load panel once
panel = load_tertiary_panel(pop_min=POP_MIN, pop_max=POP_MAX, hhi_max=DIVERSITY_HHI_MAX)
panel = panel.merge(build_acs_target(horizon_years=HORIZON_YEARS), on=['fips','period_end_date'], how='left')
panel = panel.merge(load_mf_context(), on='fips', how='left')
panel_fe, feature_cols = build_feature_matrix(panel)
panel_z = zscore_within_cohort(panel_fe, feature_cols)

print(f"Panel: {len(panel):,} rows, {len(feature_cols)} features, universe size for {TEST_YEAR} = {len(panel[panel['year']==TEST_YEAR])}")

# Run ELITE for just this test year
res = walk_forward(panel_z, panel_fe, feature_cols,
    target_col=TARGET_COL_TOTAL_RETURN, strict_pit_targets=STRICT_PIT,
    top_n=TOP_N, max_per_state=MAX_PER_STATE, elite_top_pct=ELITE_TOP_PCT,
    test_years=[TEST_YEAR])

if not res:
    raise RuntimeError(f"No result for {TEST_YEAR} — likely too few training cohorts. Try a later TEST_YEAR.")
elite_result = res[0]
elite_picks = elite_result.top5.copy()
print(f"\nELITE ran with {elite_result.n_train_cohorts} training cohorts, {elite_result.n_winners} elite winners in reference set")

Panel: 2,309 rows, 91 features, universe size for 2020 = 89

Walk-forward schedule [STRICT-PIT ELITE(25%)] (1 iterations, min_train_years=5, target=target_total_return_36mo_demeaned):
  test=2020   train_cohorts=10   span=2008-2017

ELITE ran with 10 training cohorts, 25 elite winners in reference set


## 2. Compute HighCap picks for the same test year

In [16]:
con = duckdb.connect(WAREHOUSE, read_only=True)
uni_year = con.execute(f'''
    SELECT fips, county_name, state_name, pop_latest,
           cap_rate_proxy, mf_cap_rate_estimated, employment_diversity_hhi,
           zori_forward_return_36mo
    FROM main_marts.mart_county_features
    WHERE period_end_date = '{TEST_YEAR}-12-31'
      AND pop_latest > {POP_MIN} AND pop_latest <= {POP_MAX}
      AND employment_diversity_hhi IS NOT NULL
      AND employment_diversity_hhi <= {DIVERSITY_HHI_MAX}
    ORDER BY cap_rate_proxy DESC
''').df()

# Also pull exit cap for realized TR (may be empty if TEST_YEAR + 3 > 2025)
exit_year = TEST_YEAR + HORIZON_YEARS
exit_caps = con.execute(f'''
    SELECT fips, cap_rate_proxy AS cap_exit
    FROM main_marts.mart_county_features
    WHERE period_end_date = '{exit_year}-12-31'
''').df()
con.close()

# GUARD: ensure cap_exit column exists even if exit_year has no data yet
if exit_caps.empty:
    uni_year["cap_exit"] = np.nan
    print(f"  ⚠️  No mart data for {exit_year}-12-31 (future/unobservable) → cap_exit set to NaN")
else:
    uni_year = uni_year.merge(exit_caps, on="fips", how="left")

# GUARD: ensure tr_zori36 column always exists
uni_year["tr_zori36"] = (
    (1 + uni_year["zori_forward_return_36mo"])
    * (uni_year["cap_rate_proxy"] / uni_year["cap_exit"]) - 1
)

# Apply state cap on HighCap ranking
picked, state_counts = [], {}
for _, r in uni_year.iterrows():
    if state_counts.get(r["state_name"], 0) >= MAX_PER_STATE:
        continue
    picked.append(r)
    state_counts[r["state_name"]] = state_counts.get(r["state_name"], 0) + 1
    if len(picked) >= TOP_N:
        break
highcap_picks = pd.DataFrame(picked)

print(f"HighCap picks for {TEST_YEAR}: {len(highcap_picks)}   "
      f"uni_year rows: {len(uni_year)}   tr_zori36 non-null: {uni_year['tr_zori36'].notna().sum()}")


HighCap picks for 2020: 10   uni_year rows: 89   tr_zori36 non-null: 89


## 3. Side-by-side picks

Both methods' top-N for the test year, with cap rate, MF-adjusted cap, and HHI. `⭐` = pick appears in both methods.

In [17]:
# Enrich ELITE picks with raw cap rates from mart (elite_picks has z-scored)
elite_enriched = elite_picks[['fips','county_name','state_name','pop_latest']].merge(
    uni_year[['fips','cap_rate_proxy','mf_cap_rate_estimated','employment_diversity_hhi','zori_forward_return_36mo','tr_zori36']],
    on='fips', how='left')

hc_set = set(highcap_picks['fips'])
el_set = set(elite_enriched['fips'])
intersection = hc_set & el_set

print("=" * 140)
print(f"{TEST_YEAR} TOP-{TOP_N} — pop {POP_MIN//1000}K-{POP_MAX//1000}K + HHI ≤ {DIVERSITY_HHI_MAX}")
print("=" * 140)
print(f"\n{'#':<3} {'HighCap':<38} {'cap_ACS':>8} {'cap_MF':>7} {'HHI':>7}   |   {'#':<3} {'ELITE':<38} {'cap_ACS':>8} {'cap_MF':>7} {'HHI':>7}")
print("-" * 140)
for i in range(TOP_N):
    def fmt(r):
        if r is None: return " "*72
        marker = "⭐" if r['fips'] in intersection else " "
        name = f"{marker}{r['county_name'].replace(' County','')}, {r['state_name']}"
        mf_v = r['mf_cap_rate_estimated']
        mf_str = f"{mf_v*100:>6.2f}%" if pd.notna(mf_v) else "  n/a "
        return f"{name:<38} {r['cap_rate_proxy']*100:>7.2f}% {mf_str}   {r['employment_diversity_hhi']:>6.4f}"
    hc_r = highcap_picks.iloc[i] if i < len(highcap_picks) else None
    el_r = elite_enriched.iloc[i] if i < len(elite_enriched) else None
    print(f"{i+1:<3} {fmt(hc_r)}   |   {i+1:<3} {fmt(el_r)}")

2020 TOP-10 — pop 300K-500K + HHI ≤ 0.1

#   HighCap                                 cap_ACS  cap_MF     HHI   |   #   ELITE                                   cap_ACS  cap_MF     HHI
--------------------------------------------------------------------------------------------------------------------------------------------
1    Cumberland, North Carolina               8.30%   9.00%   0.0539   |   1    Osceola, Florida                         6.83%   8.35%   0.0971
2    Nueces, Texas                            8.12%   9.91%   0.0779   |   2    St. Lucie, Florida                       6.35%   7.93%   0.0742
3    Luzerne, Pennsylvania                    7.73%   9.51%   0.0836   |   3    Seminole, Florida                        5.71%   7.23%   0.0775
4    Lubbock, Texas                           7.54%   8.50%   0.0655   |   4    Sarasota, Florida                        5.91%   7.07%   0.0922
5    Lucas, Ohio                              7.31%   8.91%   0.0846   |   5    Spartanburg, South C

## 4. Intersection — markets endorsed by both

In [18]:
print("=" * 100)
print(f"⭐ CONSENSUS PICKS — in BOTH HighCap top-{TOP_N} AND ELITE top-{TOP_N}")
print("=" * 100)
if intersection:
    both = uni_year[uni_year['fips'].isin(intersection)]
    for _, r in both.iterrows():
        mf = (r['mf_cap_rate_estimated'] or 0)*100
        print(f"  ⭐ {r['county_name']}, {r['state_name']:20s}  pop={int(r['pop_latest']):>7,}  cap_ACS={r['cap_rate_proxy']*100:5.2f}%  cap_MF={mf:5.2f}%  HHI={r['employment_diversity_hhi']:.4f}")
else:
    print("  (No consensus picks at this top-N)")
    print(f"\n  → Try raising TOP_N, or use a different pop band / HHI ceiling.")
    print(f"  → For 100-300K these two methods often have zero overlap at top-10 — they're orthogonal strategies.")

n_uni = len(uni_year)
union_size = len(hc_set | el_set)
print(f"\nOverlap: {len(intersection)}/{TOP_N}   Union: {union_size} unique counties   Universe: {n_uni}   Random expected overlap: {TOP_N*TOP_N/n_uni:.2f}")

⭐ CONSENSUS PICKS — in BOTH HighCap top-10 AND ELITE top-10
  (No consensus picks at this top-N)

  → Try raising TOP_N, or use a different pop band / HHI ceiling.
  → For 100-300K these two methods often have zero overlap at top-10 — they're orthogonal strategies.

Overlap: 0/10   Union: 20 unique counties   Universe: 89   Random expected overlap: 1.12


## 5. Realized returns (if observable)

If the test year + 3 years is in the past, we can compare against actual realized 36mo total returns.

In [23]:
import duckdb, numpy as np, pandas as pd                                                                                                                                                                            
                                        
con_x = duckdb.connect(WAREHOUSE, read_only=True)                                                                                                                                                                   
uni_year = con_x.execute(f"SELECT fips, county_name, state_name, pop_latest, cap_rate_proxy, employment_diversity_hhi, zori_forward_return_36mo FROM main_marts.mart_county_features WHERE period_end_date = 
'{TEST_YEAR}-12-31' AND pop_latest > {POP_MIN} AND pop_latest <= {POP_MAX} AND employment_diversity_hhi IS NOT NULL AND employment_diversity_hhi <= {DIVERSITY_HHI_MAX}").df()
_ec = con_x.execute(f"SELECT fips, cap_rate_proxy AS cap_exit FROM main_marts.mart_county_features WHERE period_end_date = '{TEST_YEAR + HORIZON_YEARS}-12-31'").df()
con_x.close()           
                                                                                                                                                                                                                  
if _ec.empty:           
  uni_year["cap_exit"] = np.nan                                                                                                                                                                                   
else:                                     
  uni_year = uni_year.merge(_ec, on="fips", how="left")                                                                                                                                                           
uni_year["tr_zori36"] = (1 + uni_year["zori_forward_return_36mo"]) * (uni_year["cap_rate_proxy"] / uni_year["cap_exit"]) - 1
                                                                                                                                                                                                                  
print(f"uni_year: {len(uni_year)} rows, tr_zori36 columns={list(uni_year.columns)}")
print(f"tr_zori36 non-null: {uni_year['tr_zori36'].notna().sum()}")
                                                                                                                                                                                                                  
if TEST_YEAR + HORIZON_YEARS > 2025:              
  print(f"Test year not observable yet")                                                                                                                                                                          
else:                                                                                                                                                                                                               
  uni_tr = uni_year['tr_zori36'].mean(skipna=True) * 100
  print(f"Universe mean realized TR-ZORI36 = {uni_tr:+.2f}%")

SyntaxError: unterminated f-string literal (detected at line 4) (2669613156.py, line 4)

## 6. Picks for ALL test years — both methods side-by-side

Runs both methods for every observable cohort year and shows top-N picks per year with realized returns where available.

In [19]:
# Run ELITE for all years (2015-2025) — reuse config from Section 0
print("Running ELITE for all test years 2015-2025...\n")
res_all = walk_forward(panel_z, panel_fe, feature_cols,
    target_col=TARGET_COL_TOTAL_RETURN, strict_pit_targets=STRICT_PIT,
    top_n=TOP_N, max_per_state=MAX_PER_STATE, elite_top_pct=ELITE_TOP_PCT)
print(f"\nCompleted {len(res_all)} test years")

Running ELITE for all test years 2015-2025...


Walk-forward schedule [STRICT-PIT ELITE(25%)] (21 iterations, min_train_years=5, target=target_total_return_36mo_demeaned):
  SKIP test=2005   train_cohorts= 0   span=(none)
  SKIP test=2006   train_cohorts= 0   span=(none)
  SKIP test=2007   train_cohorts= 0   span=(none)
  SKIP test=2008   train_cohorts= 0   span=(none)
  SKIP test=2009   train_cohorts= 0   span=(none)
  SKIP test=2010   train_cohorts= 0   span=(none)
  SKIP test=2011   train_cohorts= 1   span=2008-2008
  SKIP test=2012   train_cohorts= 2   span=2008-2009
  SKIP test=2013   train_cohorts= 3   span=2008-2010
  SKIP test=2014   train_cohorts= 4   span=2008-2011
  test=2015   train_cohorts= 5   span=2008-2012
  test=2016   train_cohorts= 6   span=2008-2013
  test=2017   train_cohorts= 7   span=2008-2014
  test=2018   train_cohorts= 8   span=2008-2015
  test=2019   train_cohorts= 9   span=2008-2016
  test=2020   train_cohorts=10   span=2008-2017
  test=2021   train_cohorts=

In [9]:
# Load full universe with realized returns for ALL years (not just TEST_YEAR)
con = duckdb.connect(WAREHOUSE, read_only=True)
uni_all = con.execute(f'''
    SELECT fips, county_name, state_name, pop_latest,
           EXTRACT(YEAR FROM period_end_date)::INT AS year,
           cap_rate_proxy, mf_cap_rate_estimated, employment_diversity_hhi,
           zori_forward_return_36mo
    FROM main_marts.mart_county_features
    WHERE EXTRACT(MONTH FROM period_end_date) = 12
      AND pop_latest > {POP_MIN} AND pop_latest <= {POP_MAX}
      AND employment_diversity_hhi IS NOT NULL
      AND employment_diversity_hhi <= {DIVERSITY_HHI_MAX}
''').df()
con.close()

# Attach 3yr exit cap for realized TR calc
exit_lookup = uni_all[['fips','year','cap_rate_proxy']].rename(columns={'year':'ey','cap_rate_proxy':'cap_exit'})
uni_all['ey'] = uni_all['year'] + 3
uni_all = uni_all.merge(exit_lookup, on=['fips','ey'], how='left').drop(columns=['ey'])
uni_all['tr_zori36'] = (1 + uni_all['zori_forward_return_36mo']) * (uni_all['cap_rate_proxy'] / uni_all['cap_exit']) - 1

def highcap_top_n(year, uni_df, top_n, max_per_state):
    c = uni_df[uni_df['year']==year].dropna(subset=['cap_rate_proxy']).sort_values('cap_rate_proxy', ascending=False)
    picked, sc = [], {}
    for _, r in c.iterrows():
        if sc.get(r['state_name'], 0) >= max_per_state: continue
        picked.append(r)
        sc[r['state_name']] = sc.get(r['state_name'], 0) + 1
        if len(picked) >= top_n: break
    return pd.DataFrame(picked)

print(f"Universe loaded: {len(uni_all):,} rows")

Universe loaded: 2,309 rows


### 6.1 Compact table — picks per year for both methods

In [10]:
# GUARD: rebuild tr_zori36 on uni_all if it's missing (e.g. cell 15 wasn't run yet).
if "tr_zori36" not in uni_all.columns:
    print("  ⚠️  tr_zori36 missing from uni_all — recomputing from zori_forward_return_36mo + cap_rate_proxy at Y+3...")
    exit_lookup = uni_all[["fips","year","cap_rate_proxy"]].rename(columns={"year":"ey","cap_rate_proxy":"cap_exit"})
    uni_all["ey"] = uni_all["year"] + 3
    uni_all = uni_all.merge(exit_lookup, on=["fips","ey"], how="left").drop(columns=["ey"])
    uni_all["tr_zori36"] = (1 + uni_all["zori_forward_return_36mo"]) * (uni_all["cap_rate_proxy"] / uni_all["cap_exit"]) - 1

# Compact per-year picks: one row per (year, method) with all picks as comma-separated list
rows = []
for r in res_all:
    yr = r.test_year
    if yr < 2015: continue
    elite_picks_yr = r.top5[['fips','county_name','state_name']].copy()
    hc_picks_yr = highcap_top_n(yr, uni_all, TOP_N, MAX_PER_STATE)
    if hc_picks_yr.empty: continue

    hc_names = [f"{c.replace(' County','')} {s[:2]}" for c,s in zip(hc_picks_yr['county_name'], hc_picks_yr['state_name'])]
    el_names = [f"{c.replace(' County','')} {s[:2]}" for c,s in zip(elite_picks_yr['county_name'], elite_picks_yr['state_name'])]

    hc_fips = set(hc_picks_yr['fips'])
    el_fips = set(elite_picks_yr['fips'])
    inter = hc_fips & el_fips
    inter_names = sorted(set([f"{r2['county_name'].replace(' County','')} {r2['state_name'][:2]}"
                              for _, r2 in uni_all[(uni_all['year']==yr) & (uni_all['fips'].isin(inter))].iterrows()]))

    # Realized returns if observable
    hc_tr = hc_picks_yr.merge(uni_all[uni_all['year']==yr][['fips','tr_zori36']], on='fips')['tr_zori36'].mean(skipna=True)*100
    el_tr_col = elite_picks_yr.merge(uni_all[uni_all['year']==yr][['fips','tr_zori36']], on='fips')['tr_zori36']
    el_tr = el_tr_col.mean(skipna=True)*100
    uni_tr = uni_all[uni_all['year']==yr]['tr_zori36'].mean(skipna=True)*100

    rows.append({
        'year': yr,
        'HighCap_picks': ' · '.join(hc_names),
        'ELITE_picks':   ' · '.join(el_names),
        'consensus':     ', '.join(inter_names) if inter_names else '—',
        'HC_TR': hc_tr if pd.notna(hc_tr) else np.nan,
        'EL_TR': el_tr if pd.notna(el_tr) else np.nan,
        'uni_TR': uni_tr if pd.notna(uni_tr) else np.nan,
    })

per_year_df = pd.DataFrame(rows)
per_year_df['HC_edge'] = per_year_df['HC_TR'] - per_year_df['uni_TR']
per_year_df['EL_edge'] = per_year_df['EL_TR'] - per_year_df['uni_TR']

# Compact summary table
print("=" * 130)
print(f"Per-year summary (top-{TOP_N} picks, {POP_MIN//1000}K-{POP_MAX//1000}K + HHI ≤ {DIVERSITY_HHI_MAX})")
print("=" * 130)
compact = per_year_df[['year','HC_TR','EL_TR','uni_TR','HC_edge','EL_edge','consensus']].copy()
for c in ['HC_TR','EL_TR','uni_TR','HC_edge','EL_edge']:
    compact[c] = compact[c].map(lambda x: f'{x:+.2f}' if pd.notna(x) else '  n/a  ')
print(compact.to_string(index=False))

KeyError: 'tr_zori36'

### 6.2 Full picks per year — side-by-side with hit markers

For each year: both methods' top-N picks with markers for:
- `⭐` = pick appears in BOTH methods (consensus)
- `✓` = pick actually landed in the actual top-N by realized TR (observable years only)

In [11]:
# GUARD: rebuild tr_zori36 on uni_all if it's missing (e.g. cell 15 wasn't run yet).
if "tr_zori36" not in uni_all.columns:
    print("  ⚠️  tr_zori36 missing from uni_all — recomputing from zori_forward_return_36mo + cap_rate_proxy at Y+3...")
    exit_lookup = uni_all[["fips","year","cap_rate_proxy"]].rename(columns={"year":"ey","cap_rate_proxy":"cap_exit"})
    uni_all["ey"] = uni_all["year"] + 3
    uni_all = uni_all.merge(exit_lookup, on=["fips","ey"], how="left").drop(columns=["ey"])
    uni_all["tr_zori36"] = (1 + uni_all["zori_forward_return_36mo"]) * (uni_all["cap_rate_proxy"] / uni_all["cap_exit"]) - 1

for r in res_all:
    yr = r.test_year
    if yr < 2015: continue
    elite_picks_yr = r.top5.copy()
    hc_picks_yr = highcap_top_n(yr, uni_all, TOP_N, MAX_PER_STATE)
    if hc_picks_yr.empty: continue

    hc_fips = list(hc_picks_yr['fips'])
    el_fips = list(elite_picks_yr['fips'])
    inter = set(hc_fips) & set(el_fips)

    # Actual top-N by realized TR (if observable)
    observable_year = yr + HORIZON_YEARS <= 2025
    if observable_year:
        yr_pool = uni_all[(uni_all['year']==yr) & uni_all['tr_zori36'].notna()]
        actual_top = set(yr_pool.nlargest(TOP_N, 'tr_zori36')['fips']) if not yr_pool.empty else set()
    else:
        actual_top = None

    print(f"\n===== {yr} (universe n={len(uni_all[uni_all['year']==yr])}) =====")
    obs_note = "" if actual_top is not None else "  [returns not yet observable]"
    print(f"{'#':<3}  {'HighCap':<32}  {'ELITE':<32}{obs_note}")
    for i in range(TOP_N):
        hc_r = hc_picks_yr.iloc[i] if i < len(hc_picks_yr) else None
        el_r = elite_picks_yr.iloc[i] if i < len(elite_picks_yr) else None
        def fmt(r):
            if r is None: return " "*32
            marks = ""
            if r['fips'] in inter: marks += "⭐"
            if actual_top is not None and r['fips'] in actual_top: marks += "✓"
            name = f"{r['county_name'].replace(' County','')} {r['state_name'][:2]}"
            return f"{marks}{name}"[:32].ljust(32)
        print(f"{i+1:<3}  {fmt(hc_r)}  {fmt(el_r)}")

    if observable_year:
        hc_hits = len(set(hc_fips) & actual_top)
        el_hits = len(set(el_fips) & actual_top)
        uni_tr = uni_all[uni_all['year']==yr]['tr_zori36'].mean(skipna=True)*100
        hc_tr = hc_picks_yr.merge(uni_all[uni_all['year']==yr][['fips','tr_zori36']], on='fips')['tr_zori36'].mean(skipna=True)*100
        el_tr = elite_picks_yr.merge(uni_all[uni_all['year']==yr][['fips','tr_zori36']], on='fips')['tr_zori36'].mean(skipna=True)*100
        print(f"   HC realized TR: {hc_tr:+.2f}% (edge {hc_tr-uni_tr:+.2f}pp, {hc_hits}/{TOP_N} hits)   "
              f"EL realized TR: {el_tr:+.2f}% (edge {el_tr-uni_tr:+.2f}pp, {el_hits}/{TOP_N} hits)")
    if inter:
        inter_names = sorted([f"{r2['county_name'].replace(' County','')} {r2['state_name'][:2]}"
                              for _, r2 in uni_all[(uni_all['year']==yr) & (uni_all['fips'].isin(inter))].iterrows()])
        print(f"   ⭐ Consensus: {', '.join(set(inter_names))}")


===== 2015 (universe n=89) =====
#    HighCap                           ELITE                           
1    ✓St. Lucie Fl                     Larimer Co                      
2    ✓Genesee Mi                       Charleston So                   
3    Nueces Te                         ⭐✓Lake Fl                       
4    ⭐✓Marion Fl                       Chatham Ge                      
5    Lubbock Te                        Sarasota Fl                     
6    Escambia Fl                       ⭐✓Marion Fl                     
7    Bell Te                           Manatee Fl                      
8    ⭐✓Lake Fl                         Hamilton In                     
9    Cumberland No                     Lane Or                         
10   Lucas Oh                          Northampton Pe                  


KeyError: 'tr_zori36'